# SteerMoE: Steering via Expert (De)Activation

[Paper](https://arxiv.org/abs/2509.09660) ·
[Official code](https://github.com/adobe-research/SteerMoE)

Steer OLMoE-1B-7B by deactivating the experts found in
`expert_detection.ipynb`, using EasySteer's `moe_router` algorithm in
`steermoe` mode (paper-exact mechanism: router logits are log-softmaxed,
activated experts forced to per-token max + ε, deactivated experts to
min − ε, before top-k expert selection).

Run `expert_detection.ipynb` first to produce `steermoe_digits.json`.

In [1]:
import json
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
from vllm import LLM, SamplingParams
from vllm.hidden_states import deserialize_hidden_states
from vllm.steer_vectors.request import SteerVectorRequest

MODEL = os.path.expanduser("~/models/OLMoE-1B-7B-0125-Instruct")

llm = LLM(
    model=MODEL,
    enable_steer_vector=True,
    enforce_eager=True,
    tensor_parallel_size=1,
    enable_chunked_prefill=False,
    enable_prefix_caching=False,
    gpu_memory_utilization=0.4,
    max_model_len=4096,
)
tok = llm.get_tokenizer()


def rpc(method, *args, **kwargs):
    return llm.llm_engine.collective_rpc(method, args=args, kwargs=kwargs)[0]


def gen(text, steer_req=None, max_tokens=64):
    prompt = tok.apply_chat_template(
        [{"role": "user", "content": text}], tokenize=False,
        add_generation_prompt=True)
    ids = tok(prompt, add_special_tokens=False).input_ids
    outs = llm.generate(
        {"prompt_token_ids": ids},
        sampling_params=SamplingParams(temperature=0.0,
                                       max_tokens=max_tokens),
        steer_vector_request=steer_req)
    return outs[0].outputs[0].text

/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/pydantic/dataclasses.py:313: UserWarning: `config` is set via both the `dataclass` decorator and `__pydantic_config__` for dataclass SteerVectorConfig. The `config` specification from `dataclass` decorator will take priority.
  return create_dataclass if _cls is None else create_dataclass(_cls)


INFO 08-02 18:23:24 [api_utils.py:273] non-default args: {'max_model_len': 4096, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'enforce_eager': True, 'enable_steer_vector': True, 'enable_chunked_prefill': False, 'model': '/home/xhl/models/OLMoE-1B-7B-0125-Instruct'}


INFO 08-02 18:23:24 [model.py:623] Resolved architecture: OlmoeForCausalLM


INFO 08-02 18:23:24 [model.py:1788] Using max model len 4096


...

WARNING 08-02 18:23:54 [vllm.py:1266] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


(EngineCore pid=1614056) 

INFO 08-02 18:23:54 [kernel.py:295] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


(EngineCore pid=1614056) 

INFO 08-02 18:23:54 [vllm.py:1445] Cudagraph is disabled under eager mode


## 1. Steer away from digits

Deactivating the 100 digit-linked experts flips greedy counting from
digits to written number words.

In [2]:
digit_req = SteerVectorRequest(
    "steer-away-from-digits", 1,
    steer_vector_local_path=os.path.abspath("steermoe_digits.json"),
    algorithm="moe_router",
    prefill_trigger_tokens=[-1], generate_trigger_tokens=[-1])

print("=====Baseline=====")
print(gen("Count to fifteen."))
print("=====Steered (digit experts deactivated)=====")
print(gen("Count to fifteen.", digit_req))

=====Baseline=====


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00, 20.86it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.21it/s, est. speed input: 20.50 toks/s, output: 37.38 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.79s/it, est. speed input: 9.50 toks/s, output: 35.78 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.79s/it, est. speed input: 9.50 toks/s, output: 35.78 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.79s/it, est. speed input: 9.50 toks/s, output: 35.78 toks/s]

Sure, let's count together from one to fifteen. Here's the sequence:

1 - Two - Three - Four - Five - Six - Seven - Eight - Nine - Ten - Eleven - Twelve - Thirteen - Fourteen - Fifteen

That's the full sequence from one to fifteen. How can


## 2. Verify the mechanism, not just the behavior

Capture hooks register after steering hooks, so the `router_logits`
stream records **post-steering** logits. Every deactivated expert must be
absent from every token's top-8 at its configured layer.

In [3]:
with open("steermoe_digits.json") as f:
    deact = {int(l): c["deactivate_ids"]
             for l, c in json.load(f)["layer_configs"].items()}
TOP_K = 8

prompt = tok.apply_chat_template(
    [{"role": "user", "content": "Count to fifteen."}], tokenize=False,
    add_generation_prompt=True)
ids = tok(prompt, add_special_tokens=False).input_ids

rpc("start_capture", "router_logits")
llm.generate({"prompt_token_ids": ids},
             sampling_params=SamplingParams(temperature=0.0, max_tokens=1),
             steer_vector_request=digit_req)
steered = {lid: t.float().numpy()
           for lid, t in deserialize_hidden_states(
               rpc("fetch_captured", "router_logits")).items()}
rpc("stop_capture", "router_logits")

leaks = 0
for layer, expert_ids in deact.items():
    top = np.argsort(steered[layer], axis=-1)[:, -TOP_K:]
    leaks += int(np.isin(top, expert_ids).sum())
print(f"deactivated-expert selections post-steering: {leaks} "
      f"(0 = steering is airtight)")

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00, 1544.29it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 27.06it/s, est. speed input: 460.84 toks/s, output: 27.09 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 25.96it/s, est. speed input: 460.84 toks/s, output: 27.09 toks/s]

deactivated-expert selections post-steering: 0 (0 = steering is airtight)


## 3. Faithfulness with the paper's released expert rankings

The official repo ships precomputed rankings per model and behavior.
Positive-Δ experts are linked to *faithful* answers (repeating the
document), negative-Δ experts to *unfaithful* ones (overriding the
document with parametric knowledge).

The rankings are under the
[Adobe Research License](https://github.com/adobe-research/SteerMoE/blob/main/LICENSE)
(noncommercial research), so we download them from the official repo
rather than vendoring them here.

In [4]:
import urllib.parse
import urllib.request

PKL = ("activations_[allenai--OLMoE-1B-7B-0125-Instruct]"
       "_[faithfulness].pkl")
if not os.path.exists(PKL):
    url = ("https://github.com/adobe-research/SteerMoE/raw/main/"
           "activations/" + urllib.parse.quote(PKL))
    urllib.request.urlretrieve(url, PKL)
    print("downloaded", PKL)

import pandas as pd

df = pd.read_pickle(PKL).sort_values("risk_diff_abs", ascending=False)


def deact_request(name, req_id, rows):
    cfgs = {}
    for row in rows.itertuples():
        cfgs.setdefault(str(int(row.layer)), {
            "mode": "steermoe", "deactivate_ids": []
        })["deactivate_ids"].append(int(row.expert))
    path = os.path.abspath(f"steermoe_{name}.json")
    with open(path, "w") as f:
        json.dump({"layer_configs": cfgs}, f, indent=2)
    return SteerVectorRequest(
        name, req_id, steer_vector_local_path=path,
        algorithm="moe_router",
        prefill_trigger_tokens=[-1], generate_trigger_tokens=[-1])


# Paper Table A.1 for OLMoE faithfulness: 0 activated / 50 deactivated.
# steer-faithful deactivates unfaithfulness-linked (negative-Δ) experts;
# steer-unfaithful deactivates the faithful-linked (positive-Δ) ones.
faithful_req = deact_request(
    "steer-faithful", 2, df[df.risk_diff < 0].head(50))
unfaithful_req = deact_request(
    "steer-unfaithful", 3, df[df.risk_diff > 0].head(50))

OLMoE-1B-7B-0125-Instruct is already highly faithful on short
counterfactual QA (near ceiling), so the *visible* demo-scale effect is
the reverse direction: deactivating the faithful-linked experts makes the
model override the document with parametric knowledge. (The paper's +27%
faithfulness gains are measured on benchmarks where baselines fail often —
FaithEval, CF-TriviaQA — not on ceiling-level prompts.)

In [5]:
DEMOS = [
    "Document: Romeo and Juliet was written by Jane Austen\n Question: "
    "Who wrote Romeo and Juliet? \n Final Answer Only:",
    "Document: The sun rises in the west\n Question: In which direction "
    "does the sun rise? \n Final Answer Only:",
    "Document: The capital of France is Lyon\n Question: What is the "
    "capital of France? \n Final Answer Only:",
]
for demo in DEMOS:
    print("Q:", demo.splitlines()[0])
    print("  baseline        :", gen(demo))
    print("  steer-faithful  :", gen(demo, faithful_req))
    print("  steer-unfaithful:", gen(demo, unfaithful_req))

Q: Document: Romeo and Juliet was written by Jane Austen


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00, 510.19it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  6.82it/s, est. speed input: 272.93 toks/s, output: 27.29 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  3.77it/s, est. speed input: 135.62 toks/s, output: 30.14 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  3.77it/s, est. speed input: 135.62 toks/s, output: 30.14 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  3.74it/s, est. speed input: 135.62 toks/s, output: 30.14 toks/s]

  steer-unfaithful: The capital of France is Lyon.
